# Phase 4, Sub-BCS Stacking Method
---
Unlike the previous phases, this approach first models each anatomical
region independently. The regional predictions are combined, enabling
the meta-model to learn the relative importance of each region.

Base models: Ridge, Lasso, Ordinal Logistic Regression.
Meta-model: Ridge regression.
Metrics: R², MSE, MAE, clinical tolerance ±0.5 and ±1.0 BCS.

In [1]:
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import RepeatedKFold, GridSearchCV, cross_val_predict, KFold
from sklearn.base import clone, BaseEstimator, RegressorMixin
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge, Lasso
from mord import LogisticAT

warnings.filterwarnings('ignore')

In [2]:
df = pd.read_excel('../COMBINED_HORSES_DATA.xlsx')

# optimal features selected by forward sequential feature selection
optimal_features = ['beta_1', 'beta_3', 'beta_5', 'beta_7', 'beta_8', 'beta_23', 'beta_31']

X_opt = df[optimal_features]
y_total = df['BCS']
sub_bcs = ['BCS_Neck', 'BCS_Withers', 'BCS_Back', 'BCS_Tailhead',
           'BCS_Shoulder', 'BCS_Ribs', 'BCS_Thighs']

print(f"Dataset: {X_opt.shape[0]} samples, {X_opt.shape[1]} features")
print(f"Sub-BCS Regions: {len(sub_bcs)}")

Dataset: 208 samples, 7 features
Sub-BCS Regions: 7


In [3]:
# sklearn-compatible wrapper for mord ordinal regression with non-consecutive label support
class OrdinalWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, alpha=0.1):
        self.alpha = alpha

    # fit ordinal model using a remapped integer encoding of labels
    def fit(self, X, y):
        unique_vals = np.sort(np.unique(y))
        self.label_map_ = {v: i for i, v in enumerate(unique_vals)}
        self.inverse_map_ = {i: v for v, i in self.label_map_.items()}
        y_enc = np.array([self.label_map_[v] for v in y])
        self.model_ = LogisticAT(alpha=self.alpha)
        self.model_.fit(X, y_enc)
        return self

    # decode predicted integer labels back to original BCS values
    def predict(self, X):
        y_enc = self.model_.predict(X)
        return np.array([self.inverse_map_[int(round(v))] for v in y_enc])

## Stacking with nested CV
---
Hyperparameters are tuned INSIDE each outer fold (inner 5-fold CV
repeated 3 times). The test fold never influences hyperparameter
selection. Out-of-fold predictions prevent leakage in the
meta-feature matrix Z.

In [ ]:
PARAM_GRIDS = {
    'Ridge':   {'alpha': np.logspace(-3, 3, 60)},
    'Lasso':   {'alpha': np.logspace(-4, 1, 60)},
    'Ordinal': {'alpha': [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0]},
}

BASE_MODELS = {
    'Ridge':   lambda: Ridge(random_state=42),
    'Lasso':   lambda: Lasso(random_state=42, max_iter=10_000),
    'Ordinal': lambda: OrdinalWrapper(),
}

meta_model = Ridge(alpha=1.0, random_state=42)
outer_rkf = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)
inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)

print("Stacking Results, Nested CV")
print("Base Model -> 7 Sub-BCS Predictions -> Ridge Meta-Model -> Total BCS")
print()

meta_weights = {}

for name, base_func in BASE_MODELS.items():
    cv_r2, cv_mse, cv_mae, cv_acc05, cv_acc10 = [], [], [], [], []
    fold_weights = []

    region_mse = {col: [] for col in sub_bcs}
    region_mae = {col: [] for col in sub_bcs}
    region_r2 = {col: [] for col in sub_bcs}

    for train_idx, test_idx in outer_rkf.split(X_opt):
        X_train = X_opt.iloc[train_idx]
        X_test = X_opt.iloc[test_idx]
        y_train = y_total.iloc[train_idx]
        y_test = y_total.iloc[test_idx]

        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)

        gs = GridSearchCV(base_func(), PARAM_GRIDS[name], cv=inner_cv,
                          scoring='neg_mean_squared_error', n_jobs=-1)
        gs.fit(X_train_s, y_train)
        best_base = gs.best_estimator_

        Z_train = np.zeros((len(X_train), len(sub_bcs)))
        Z_test = np.zeros((len(X_test), len(sub_bcs)))

        for j, col in enumerate(sub_bcs):
            y_part = df[col].iloc[train_idx].values
            y_part_test = df[col].iloc[test_idx].values

            m_oof = clone(best_base)
            Z_train[:, j] = cross_val_predict(m_oof, X_train_s, y_part, cv=5)

            m_full = clone(best_base)
            m_full.fit(X_train_s, y_part)
            Z_test[:, j] = m_full.predict(X_test_s)

            # collect per-region metrics for this fold
            region_mse[col].append(mean_squared_error(y_part_test, Z_test[:, j]))
            region_mae[col].append(mean_absolute_error(y_part_test, Z_test[:, j]))
            region_r2[col].append(r2_score(y_part_test, Z_test[:, j]))
            

        meta = clone(meta_model)
        meta.fit(Z_train, y_train)
        y_pred = meta.predict(Z_test)

        fold_weights.append(meta.coef_)

        cv_r2.append(r2_score(y_test, y_pred))
        cv_mse.append(mean_squared_error(y_test, y_pred))
        cv_mae.append(mean_absolute_error(y_test, y_pred))
        errors = np.abs(y_test.values - y_pred)
        cv_acc05.append(np.mean(errors <= 0.5))
        cv_acc10.append(np.mean(errors <= 1.0))

    meta_weights[name] = np.mean(fold_weights, axis=0)

    # per-region summary table before stacking summary
    region_labels = ['Neck', 'Withers', 'Back', 'Tailhead', 'Shoulder', 'Ribs', 'Thighs']
    print(f"{name} — Per-Region Base Model Performance (averaged over folds)")
    print(f"  {'Region':<12} {'MSE':>7} {'MAE':>7} {'R²':>7}")
    print(f"  {'-'*37}")
    for col, label in zip(sub_bcs, region_labels):
        print(f"  {label:<12} {np.mean(region_mse[col]):>7.4f} "
              f"{np.mean(region_mae[col]):>7.4f} {np.mean(region_r2[col]):>7.4f}")
    print()

    print(f"{name} — Stacking Ensemble (Total BCS)")
    print(f"  Test R²:  {np.mean(cv_r2):.4f}")
    print(f"  Test MSE: {np.mean(cv_mse):.4f}")
    print(f"  Test MAE: {np.mean(cv_mae):.4f}")
    print(f"  ±0.5 BCS: {np.mean(cv_acc05):.1%}")
    print(f"  ±1.0 BCS: {np.mean(cv_acc10):.1%}")
    print()

Stacking Results, Nested CV
Base Model -> 7 Sub-BCS Predictions -> Ridge Meta-Model -> Total BCS

Ridge — Per-Region Base Model Performance (averaged over folds)
  Region           MSE     MAE      R²
  -------------------------------------
  Neck          0.5435  0.5917  0.3060
  Withers       0.4743  0.5414  0.2925
  Back          0.4838  0.5504  0.2742
  Tailhead      0.5937  0.5875  0.3309
  Shoulder      0.4161  0.5201  0.3951
  Ribs          0.7709  0.6772  0.3039
  Thighs        0.0987  0.1109 -0.1100

Ridge — Stacking Ensemble (Total BCS)
  Test R²:  0.3886
  Test MSE: 0.4511
  Test MAE: 0.5274
  ±0.5 BCS: 58.7%
  ±1.0 BCS: 85.5%

Lasso — Per-Region Base Model Performance (averaged over folds)
  Region           MSE     MAE      R²
  -------------------------------------
  Neck          0.5441  0.5921  0.3043
  Withers       0.4750  0.5426  0.2898
  Back          0.4839  0.5503  0.2731
  Tailhead      0.5937  0.5879  0.3303
  Shoulder      0.4172  0.5208  0.3927
  Ribs         

## Meta-model weights, anatomical region importance
---
The Ridge meta-model learns which sub-BCS regions drive the
total BCS prediction. Weights are averaged over all outer folds.

In [ ]:
region_labels = ['Neck', 'Withers', 'Back', 'Tailhead', 'Shoulder', 'Ribs', 'Thighs']

for name, weights in meta_weights.items():
    abs_w = np.abs(weights)
    pct = 100 * abs_w / abs_w.sum()

    sorted_idx = np.argsort(pct)
    sorted_labels = [region_labels[i] for i in sorted_idx]
    sorted_pct = pct[sorted_idx]

    plt.figure(figsize=(10, 5))
    sns.set_theme(style="whitegrid")
    colours = sns.color_palette("Blues", len(sorted_pct))
    bars = plt.barh(sorted_labels, sorted_pct, color=colours, edgecolor='black', alpha=0.9)

    for bar in bars:
        w = bar.get_width()
        plt.text(w + 0.5, bar.get_y() + bar.get_height() / 2,
                 f'{w:.1f}%', ha='left', va='center', fontsize=11, fontweight='bold')

    plt.title(f'meta-model weights: {name.lower()} base model', fontsize=14, pad=15)
    plt.xlabel('relative importance (%)', fontsize=12)
    plt.xlim(0, max(sorted_pct) + 12)
    plt.grid(axis='x', linestyle='--', alpha=0.5)
    sns.despine()
    plt.tight_layout()
    plt.show()

    print(f"{name} Weights:")
    for lbl, p in zip(region_labels, pct):
        print(f"  {lbl:<12} {p:5.1f}%")
    print()